In [3]:
silver_table = spark.read.table("silver_accepted_loans")

print(f"Rows    : {silver_table.count():,}")
print(f"Columns : {len(silver_table.columns)}")

StatementMeta(, 2c0874cb-dc22-40d0-913c-beffe72073d7, 5, Finished, Available, Finished, False)

Rows    : 2,260,413
Columns : 104


In [3]:
from pyspark.sql import functions as F

silver_df = spark.read.table("silver_accepted_loans")

print(silver_df.count())
print(len(silver_df.columns))

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 5, Finished, Available, Finished, False)

2260413
104


**Grain:**
One row in FactLoan represents one funded LendingClub loan.

**Dimensions:**
Date
Borrower
Location
Purpose
Grade

**Fact:**
Loan

In [4]:
fact_columns = [

    # Primary Key
    "id",

    # Foreign Keys
    "issue_d",
    "grade",
    "purpose",
    "addr_state",

    # Loan Measures
    "loan_amnt",
    "funded_amnt",
    "funded_amnt_inv",
    "int_rate",
    "installment",

    # Borrower Measures
    "annual_inc",
    "dti",

    # Credit Measures
    "fico_range_low",
    "fico_range_high",

    # Performance
    "loan_status",
    "out_prncp",
    "total_pymnt",
    "total_rec_prncp",
    "total_rec_int",
    "recoveries",

    # Counts
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "total_acc",
    "revol_bal",
    "revol_util"

]

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 6, Finished, Available, Finished, False)

In [5]:
len(fact_columns)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 7, Finished, Available, Finished, False)

26

In [1]:
from pyspark.sql.functions import broadcast

dim_grade = spark.read.table("gold_dim_grade")
dim_purpose = spark.read.table("gold_dim_purpose")
dim_state = spark.read.table("gold_dim_state")
dim_status = spark.read.table("gold_dim_status")
dim_date = spark.read.table("gold_dim_date")

StatementMeta(, 2c0874cb-dc22-40d0-913c-beffe72073d7, 3, Finished, Available, Finished, False)

In [4]:
fact_loan_df = (
    silver_table.alias("f")
    .join(
        broadcast(dim_grade.alias("g")),
        "grade",
        "left"
    )
    .join(
        broadcast(dim_purpose.alias("p")),
        "purpose",
        "left"
    )
    .join(
        broadcast(dim_state.alias("s")),
        "addr_state",
        "left"
    )
    .join(
        broadcast(dim_status.alias("st")),
        "loan_status",
        "left"
    )
    .join(
        broadcast(dim_date.alias("d")),
        silver_table.issue_d == dim_date.issue_date,
        "left"
    )
)

StatementMeta(, 2c0874cb-dc22-40d0-913c-beffe72073d7, 6, Finished, Available, Finished, False)

In [5]:
fact_loan_df = fact_loan_df.select(

    "id",

    "date_key",
    "grade_key",
    "purpose_key",
    "state_key",
    "status_key",

    "loan_amnt",
    "funded_amnt",
    "funded_amnt_inv",
    "int_rate",
    "installment",

    "annual_inc",
    "dti",

    "fico_range_low",
    "fico_range_high",

    "loan_status",

    "out_prncp",
    "total_pymnt",
    "total_rec_prncp",
    "total_rec_int",

    "inq_last_6mths",
    "open_acc",
    "revol_bal",
    "revol_util",
    "total_acc",
    "recoveries"
)

StatementMeta(, 2c0874cb-dc22-40d0-913c-beffe72073d7, 7, Finished, Available, Finished, False)

In [6]:
fact_loan_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold_fact_loan")

StatementMeta(, 2c0874cb-dc22-40d0-913c-beffe72073d7, 8, Finished, Available, Finished, False)

In [7]:
gold_fact = spark.read.table("gold_fact_loan")

print(gold_fact.count())
print(len(gold_fact.columns))

display(gold_fact.limit(5))

StatementMeta(, 2c0874cb-dc22-40d0-913c-beffe72073d7, 9, Finished, Available, Finished, False)

11302065
26


SynapseWidget(Synapse.DataFrame, 3c17bb23-a72a-42bf-b3a2-39a35f86cd01)

In [10]:
from pyspark.sql import functions as F

dim_date = (
    silver_table
        .select("issue_d")
        .distinct()
        .filter(F.col("issue_d").isNotNull())
        .withColumn("issue_date", F.to_date("issue_d", "MMM-yyyy"))
        .withColumn("Year", F.year("issue_date"))
        .withColumn("Month", F.month("issue_date"))
        .withColumn("MonthName", F.date_format("issue_date", "MMMM"))
        .withColumn("Quarter", F.quarter("issue_date"))
)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 12, Finished, Available, Finished, False)

In [11]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.orderBy("issue_date")

dim_date = (
    dim_date
        .withColumn(
            "date_key",
            row_number().over(window_spec)
        )
        .select(
            "date_key",
            "issue_date",
            "Year",
            "Quarter",
            "Month",
            "MonthName"
        )
)

dim_date.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.format("delta") \
.saveAsTable("gold_dim_date")

display(dim_date)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4ec04f43-3220-42fe-91ce-a9eac3c982da)

In [12]:
dim_date.orderBy("issue_date").show(20, truncate=False)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 14, Finished, Available, Finished, False)

+--------+----------+----+-------+-----+---------+
|date_key|issue_date|Year|Quarter|Month|MonthName|
+--------+----------+----+-------+-----+---------+
|1       |2007-06-01|2007|2      |6    |June     |
|2       |2007-07-01|2007|3      |7    |July     |
|3       |2007-08-01|2007|3      |8    |August   |
|4       |2007-09-01|2007|3      |9    |September|
|5       |2007-10-01|2007|4      |10   |October  |
|6       |2007-11-01|2007|4      |11   |November |
|7       |2007-12-01|2007|4      |12   |December |
|8       |2008-01-01|2008|1      |1    |January  |
|9       |2008-02-01|2008|1      |2    |February |
|10      |2008-03-01|2008|1      |3    |March    |
|11      |2008-04-01|2008|2      |4    |April    |
|12      |2008-05-01|2008|2      |5    |May      |
|13      |2008-06-01|2008|2      |6    |June     |
|14      |2008-07-01|2008|3      |7    |July     |
|15      |2008-08-01|2008|3      |8    |August   |
|16      |2008-09-01|2008|3      |9    |September|
|17      |2008-10-01|2008|4    

In [13]:
gold_dim_date = spark.read.table("gold_dim_date")

print(gold_dim_date.count())

display(gold_dim_date.orderBy("issue_date"))

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 15, Finished, Available, Finished, False)

139


SynapseWidget(Synapse.DataFrame, a6001b24-7cc1-4722-b25d-aad23dcb1611)

In [14]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.orderBy("grade", "sub_grade")

dim_grade = (
    silver_table
    .select(
        "grade",
        "sub_grade"
    )
    .distinct()
    .withColumn(
        "grade_key",
        row_number().over(window_spec)
    )
    .select(
        "grade_key",
        "grade",
        "sub_grade"
    )
)

display(dim_grade)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6d8f1ec0-e9c9-4c43-8bcf-4722c5ddf963)

In [15]:
display(dim_grade)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7b3706b2-a387-4d02-9eef-67deb1415020)

In [16]:
dim_grade.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.format("delta") \
.saveAsTable("gold_dim_grade")

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 18, Finished, Available, Finished, False)

In [17]:
display(
    spark.read.table("gold_dim_grade")
)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 94cb96cc-1d2c-44eb-a293-1784d1e084db)

In [18]:
silver_table.groupBy("purpose") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=False)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 20, Finished, Available, Finished, False)

+------------------+-------+
|purpose           |count  |
+------------------+-------+
|debt_consolidation|1277790|
|credit_card       |516926 |
|home_improvement  |150440 |
|other             |139413 |
|major_purchase    |50429  |
|medical           |27481  |
|small_business    |24659  |
|car               |24009  |
|vacation          |15525  |
|moving            |15402  |
|house             |14131  |
|wedding           |2351   |
|renewable_energy  |1445   |
|educational       |412    |
+------------------+-------+



In [19]:
valid_purposes = [
    "debt_consolidation",
    "credit_card",
    "home_improvement",
    "other",
    "major_purchase",
    "medical",
    "small_business",
    "car",
    "vacation",
    "moving",
    "house",
    "wedding",
    "renewable_energy",
    "educational"
]

bad_rows = silver_table.filter(
    ~F.col("purpose").isin(valid_purposes)
)

print(bad_rows.count())

bad_rows.select(
    "id",
    "purpose",
    "loan_status",
    "home_ownership",
    "verification_status"
).show(50, truncate=False)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 21, Finished, Available, Finished, False)

0
+---+-------+-----------+--------------+-------------------+
|id |purpose|loan_status|home_ownership|verification_status|
+---+-------+-----------+--------------+-------------------+
+---+-------+-----------+--------------+-------------------+



In [20]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.orderBy("purpose")

dim_purpose = (
    silver_table
    .select("purpose")
    .distinct()
    .withColumn(
        "purpose_key",
        row_number().over(window_spec)
    )
    .select(
        "purpose_key",
        "purpose"
    )
)

dim_purpose.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.format("delta") \
.saveAsTable("gold_dim_purpose")

display(dim_purpose)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 50ad4f49-f704-43f3-ba06-a6ccefeb2479)

In [21]:
display(dim_purpose)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4461bb02-a0ff-428e-9095-6f28467a2de6)

In [22]:
gold_dim_purpose = spark.read.table("gold_dim_purpose")

print(gold_dim_purpose.count())

display(gold_dim_purpose)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 24, Finished, Available, Finished, False)

14


SynapseWidget(Synapse.DataFrame, 3d6983b2-b9e9-406d-bec5-472d6ea1699e)

In [23]:
window_spec = Window.orderBy("addr_state")

dim_state = (
    silver_table
    .select("addr_state")
    .distinct()
    .withColumn(
        "state_key",
        row_number().over(window_spec)
    )
    .select(
        "state_key",
        "addr_state"
    )
)

dim_state.write \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.format("delta") \
.saveAsTable("gold_dim_state")

display(dim_state)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3c40dd29-dcff-446d-aa4c-dac3e2c23df4)

In [24]:
gold_dim_state = spark.read.table("gold_dim_state")

print(gold_dim_state.count())

display(gold_dim_state)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 26, Finished, Available, Finished, False)

52


SynapseWidget(Synapse.DataFrame, 2a111987-10a6-49a1-a782-1406972cb144)

In [25]:
window_spec = Window.orderBy("loan_status")

dim_status = (
    silver_table
    .select("loan_status")
    .distinct()
    .withColumn(
        "status_key",
        row_number().over(window_spec)
    )
    .select(
        "status_key",
        "loan_status"
    )
)

dim_status.write \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.format("delta") \
.saveAsTable("gold_dim_status")

display(dim_status)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 86d1e190-260f-4270-9917-e04ec1b82522)

In [26]:
tables = [
    "gold_dim_date",
    "gold_dim_grade",
    "gold_dim_purpose",
    "gold_dim_state",
    "gold_dim_status"
]

for t in tables:
    df = spark.read.table(t)
    print(f"\n{t}")
    df.show(5, truncate=False)

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 28, Finished, Available, Finished, False)


gold_dim_date
+--------+----------+----+-------+-----+---------+
|date_key|issue_date|Year|Quarter|Month|MonthName|
+--------+----------+----+-------+-----+---------+
|1       |2007-06-01|2007|2      |6    |June     |
|2       |2007-07-01|2007|3      |7    |July     |
|3       |2007-08-01|2007|3      |8    |August   |
|4       |2007-09-01|2007|3      |9    |September|
|5       |2007-10-01|2007|4      |10   |October  |
+--------+----------+----+-------+-----+---------+
only showing top 5 rows


gold_dim_grade
+---------+-----+---------+
|grade_key|grade|sub_grade|
+---------+-----+---------+
|1        |A    |A1       |
|2        |A    |A2       |
|3        |A    |A3       |
|4        |A    |A4       |
|5        |A    |A5       |
+---------+-----+---------+
only showing top 5 rows


gold_dim_purpose
+-----------+------------------+
|purpose_key|purpose           |
+-----------+------------------+
|1          |car               |
|2          |credit_card       |
|3          |debt_consoli

In [27]:
fact = spark.read.table("gold_fact_loan")

print("Rows        :", fact.count())
print("Distinct IDs:", fact.select("id").distinct().count())

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 29, Finished, Available, Finished, False)

Rows        : 2260413
Distinct IDs: 2260413


In [28]:
from pyspark.sql.functions import current_timestamp

log_df = spark.createDataFrame([
    (
        "pl_loan_portfolio_etl",
        "nb_build_gold",
        "SUCCESS",
        gold_fact.count()
    )
], ["pipeline_name", "notebook_name", "status", "rows_processed"])

log_df = log_df.withColumn("run_time", current_timestamp())

log_df.write.mode("append").saveAsTable("audit_log")

StatementMeta(, b4308947-1c76-431c-a025-f9bc8536b474, 30, Finished, Available, Finished, False)

### Referential integrity **(orphan key)** validation

In [13]:
fact = spark.read.table("gold_fact_loan")
dim = spark.read.table("gold_dim_grade")
orphan = fact.join(
    dim,
    on="grade_key",
    how="left_anti"
)
print("Orphan Grade Keys:", orphan.count())

dim = spark.read.table("gold_dim_purpose")
orphan = fact.join(
    dim,
    on="purpose_key",
    how="left_anti"
)
print("Orphan Purpose Keys:", orphan.count())

dim = spark.read.table("gold_dim_state")
orphan = fact.join(
    dim,
    on="state_key",
    how="left_anti"
)
print("Orphan State Keys:", orphan.count())

dim = spark.read.table("gold_dim_status")
orphan = fact.join(
    dim,
    on="status_key",
    how="left_anti"
)
print("Orphan Status Keys:", orphan.count())

dim = spark.read.table("gold_dim_date")
orphan = fact.join(
    dim,
    on="date_key",
    how="left_anti"
)
print("Orphan Date Keys:", orphan.count())

StatementMeta(, 2c0874cb-dc22-40d0-913c-beffe72073d7, 15, Finished, Available, Finished, False)

Orphan Grade Keys: 0
Orphan Purpose Keys: 0
Orphan State Keys: 0
Orphan Status Keys: 0
Orphan Date Keys: 0


### Referential integrity **(orphan key)** validation